# Despliegue en producción

En esta parte del proyecto, aprenderemos a lanzar nuestro modelo a producción. En la lección anterior aprendimos cómo realizar inferencia usando el método predict. Sin embargo, tenemos una gran limitante:

*Sólo podemos hacerlo adentro de nuestro Jupyter Notebook*

Veamos primero cómo utilizar nuestro modelo entrenado en algún lugar fuera de Jupyter. Específicamente queremos que cualquier persona o dispositivo tenga la posibilidad de usar nuestro modelo para crear una predicción. 

Desarrollaremos un API REST que expondrá un endpoint que permitirá realizar predicciones. Usaremos Flask para crear nuestro API.

## API

Nuestro API será muy sencillo, pero poderoso. Crearemos un archivo app.py vacío y comencemos.

```python
# importar paquetes en app.py
from flask import Flask, request, jsonify
import pickle
import numpy as np

## Cargar el modelo

Usaremos Flask para crear y exponer endpoints, numpy para alimentar los datos al modelo y pickle para cargar el modelo en nuestro API.

En el notebook pasado guardamos el modelo usando Pickle. Ahora haremos el proceso inverso: leeremos el archivo que guardamos.


```python
modelo = None

# Cargar el modelo
with open("models/modelo.pkl", 'rb') as file:
    modelo = pickle.load(file)


## Crear app de Flask

Crearemos una app de flask con un sólo endpoint /predecir el cual será accesible a través de un POST. Los datos de entrada se enviarán en formato JSON. 

```python

app = Flask(__name__)

@app.route('/predecir', methods=['POST'])
def predict():
    # obtener json
    data = request.get_json(force=True)
    
    # convertir los datos a un arreglo de Numpy
    input_data = np.array(data['input']).reshape(1, -1)
    
    # Hacer predicción
    prediccion = modelo.predict(input_data)
    
    # regresar predicción en formato json
    return jsonify({'prediccion': int(prediccion[0])})

if __name__ == '__main__':
    app.run(debug=True)

## Ejecución

En una nueva terminal, navegamos al directorio del proyecto. Ejecutamos el API de flask con:

```python
python app.py
```

Ahora necesitamos hacer peticiones POST a este API que se está ecutando en la dirección 127.0.0.1 en el puerto 5000. Para esto, usaremos el comando curl.

Abrimos una nueva terminal (asegurandonos de no cerrar la terminal que está ejecutando el API de Flask porque eso frenaría el servidor).

Y ejecutamos el siguiente comando:

```python
curl -X POST http://127.0.0.1:5000/predecir -H "Content-Type: application/json" -d "{\"input\": [0,0,0,0,0,0,0]}"
